# OpenCode Go — Provider Test

Testing OpenCode Go as the LLM provider for lerim:
- **Lead agent**: MiniMax M2.7 (via Messages endpoint)
- **Codex sub-agent**: Kimi K2.5 (via Chat Completions endpoint)
- **DSPy pipelines**: MiniMax M2.5 (via Messages endpoint — separate from agent SDK)

In [1]:
import os, sys, json, tempfile, time
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path("/Users/kargarisaac/codes/personal/lerim/lerim-cli/.env"))

from agents import Agent, Runner, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel
from agents.extensions.experimental.codex import codex_tool, ThreadOptions, CodexOptions
set_tracing_disabled(disabled=True)

sys.path.insert(0, str(Path("/Users/kargarisaac/codes/personal/lerim/lerim-cli/src")))

OPENCODE_API_KEY = os.environ.get("OPENCODE_API_KEY", "")
print(f"OPENCODE_API_KEY: {'set (' + OPENCODE_API_KEY[:12] + '...)' if OPENCODE_API_KEY else 'NOT SET'}")

OPENCODE_API_KEY: set (sk-TIZ3IQPU0...)


## Test 1 — MiniMax M2.7 as Lead Agent

In [3]:
GO_BASE = "https://opencode.ai/zen/go"

# MiniMax M2.7 for lead agent (Messages endpoint → anthropic/ prefix)
lead_model = LitellmModel(
	model="anthropic/minimax-m2.7",
	api_key=OPENCODE_API_KEY,
	base_url=GO_BASE,
)

agent = Agent(
	name="LeadTest",
	instructions="Respond concisely in 1 sentence.",
	model=lead_model,
)
result = await Runner.run(agent, "What is 2+2?")
print(f"MiniMax M2.7 (lead): {result.final_output}")


Provider List: https://docs.litellm.ai/docs/providers

MiniMax M2.7 (lead): 4


## Test 2 — Kimi K2.5 as Codex sub-agent

Kimi uses the Chat Completions endpoint — our ResponsesProxy handles translation.

In [5]:
from lerim.runtime.responses_proxy import ResponsesProxy

test_dir = Path(tempfile.mkdtemp(prefix="opencode-go-"))
(test_dir / "hello.txt").write_text("The secret answer is 42.")

# Kimi K2.5 for Codex (Chat Completions → ResponsesProxy)
proxy = ResponsesProxy(
	backend_url=GO_BASE,
	api_key=OPENCODE_API_KEY,
)
proxy.start()
print(f"Proxy: {proxy.url} → {GO_BASE}")

agent = Agent(
	name="CodexKimiTest",
	instructions=f"Use codex to read {test_dir}/hello.txt and tell me what it says.",
	model=lead_model,  # outer agent = M2.7
	tools=[
		codex_tool(
			codex_options=CodexOptions(
				base_url=proxy.url,
				api_key="proxy-managed",
			),
			sandbox_mode="read-only",
			working_directory=str(test_dir),
			skip_git_repo_check=True,
			default_thread_options=ThreadOptions(
				model="kimi-k2.5",  # Codex uses Kimi
				approval_policy="never",
				network_access_enabled=False,
			),
		),
	],
)

try:
	result = await Runner.run(agent, "Read hello.txt and tell me what it says.", max_turns=50)
	print(f"Response: {result.final_output}")
	if "42" in str(result.final_output):
		print("Lead (M2.7) + Codex (Kimi K2.5) works!")
except Exception as e:
	print(f"Failed: {e}")
finally:
	proxy.stop()
	print("Proxy stopped")

import shutil
shutil.rmtree(test_dir, ignore_errors=True)

Proxy: http://127.0.0.1:57950 → https://opencode.ai/zen/go

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

Response: I apologize, but I'm encountering an error with the Codex tool (receiving a 403 Forbidden error). This appears to be a temporary service issue preventing me from accessing the file.

Could you please:
1. **Try again in a few moments**, or
2. **Share the contents of the file directly** so I can help you with it?
Proxy stopped


## Cleanup

In [6]:
import shutil
shutil.rmtree(test_dir, ignore_errors=True)
print("Done!")
print("\nSummary:")
print("  - OpenCode Go works as LitellmModel provider (GLM-5, Kimi K2.5 via chat/completions)")
print("  - Codex tool works via ResponsesProxy (Go has no /responses endpoint)")
print("  - Free Zen models also work (gpt-5-nano, etc.)")
print("  - MiniMax M2.7/M2.5 on Go use messages endpoint (Anthropic format — needs different handling)")

Done!

Summary:
  - OpenCode Go works as LitellmModel provider (GLM-5, Kimi K2.5 via chat/completions)
  - Codex tool works via ResponsesProxy (Go has no /responses endpoint)
  - Free Zen models also work (gpt-5-nano, etc.)
  - MiniMax M2.7/M2.5 on Go use messages endpoint (Anthropic format — needs different handling)
